This notebook is based on the code of:
V. S. F. Garnot and L. Landrieu, “Panoptic Segmentation of Satellite Image Time Series with Convolutional Temporal Attention Networks,” in Proceedings of the IEEE/CVF International Conference on Computer Vision (ICCV), 2021, pp. 4872–4881, doi: 10.1109/ICCV48922.2021.00483.

In [1]:
import json
import os
import numpy as np
from dataclasses import dataclass, field
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import utaeT
from utils import iterate, save_results, prepare_output, pad_collate, get_ntrainparams, load_config, Config, TreeDataset

In [ ]:
@dataclass
class ConfigTestPaths:
    # Dataset / paths
    config_path: Path = field(default_factory=lambda: Path.cwd() / "results_train" / "conf.json")
    model_dir: Path = field(default_factory=lambda: Path.cwd() / "results_train")
    res_dir: Path = field(default_factory=lambda: Path.cwd() / "results_test" / "vienna")

    path_base: Path = Path.cwd().parents[0] / "data" / "preprocessed data"
    paths_sentinel_data: list = None
    paths_tree_data: list = None
    paths_date_data: list = None
    def __post_init__(self):
        self.paths_sentinel_data = [
            self.path_base / "satellite_patches_vie.npy",
        ]
        self.paths_tree_data = [
            self.path_base / "tree_patches_vie.npy",
        ]
        self.paths_date_data = [
            self.path_base / "date_patches_vie.npy",
        ]

In [3]:
@dataclass
class ConfigTest:
    rdm_seed = 42

    # computation and display
    display_step: int = 1
    num_workers: int = 4
    batch_size: int = 4
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    def torch_device(self):
        return torch.device(self.device)

    fold: int = 1


In [4]:
def main(config_train, config_test,config_test_paths):
    # set random seed and device
    np.random.seed(config_train.rdm_seed)
    torch.manual_seed(config_train.rdm_seed)
    device = torch.device(config_test.device)

    sentinel_list = []
    tree_list = []
    date_list = []
    
    for p_s, p_t, p_d in zip(config_test_paths.paths_sentinel_data,config_test_paths.paths_tree_data,config_test_paths.paths_date_data):
        sentinel_list.append(np.load(p_s, mmap_mode='r'))
        tree_list.append(np.load(p_t, mmap_mode='r'))
        date_list.append(np.load(p_d, mmap_mode='r'))

    # merge all cities
    test_x = np.concatenate(sentinel_list, axis=0)
    test_y = np.concatenate(tree_list, axis=0)
    test_d = np.concatenate(date_list, axis=0)

    dt_test  = TreeDataset(test_x, test_y, test_d)

    # create dataloaders using padding
    collate_fn = lambda x: pad_collate(x, pad_value=config_train.pad_value)
    test_loader = DataLoader(
        dt_test,
        batch_size=config_test.batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=collate_fn,
    )

    print(f"Test {len(dt_test)}")

    # load model structure
    model = utaeT.UTAE(config_train)
    model = model.to(device)

    config_test.N_params = get_ntrainparams(model)
    print(model)
    print("TOTAL TRAINABLE PARAMETERS :", config_test.N_params)

    # create lists for test evaluation
    all_preds_folds = []
    all_targets_folds = []

    # define which folds should be tested
    if config_test.fold is not None:
        folds_to_run = [config_test.fold - 1]
    else:
        folds_to_run = range(5)

    # prepare output folders
    prepare_output(config_test_paths,config_test)

    for fold in folds_to_run:
        # Load weights
        sd = torch.load(
            os.path.join(config_test_paths.model_dir, "Fold_{}".format(fold+1), "model.pth.tar"),
            map_location=device,
        )
        model.load_state_dict(sd["state_dict"])
    
        print(f"\n=== Running Fold {fold+1} ===")
        
        # Inference
        print("Testing . . .")
        model.eval()
        test_metrics, preds, targets, probs, imgs = iterate(
            model,
            data_loader=test_loader,
            config_train = config_train,
            config_test = config_test,
            optimizer=None,
            mode="test",
            img_output=True
        )

        # Print and save results
        all_preds_folds.extend(preds)
        all_targets_folds.extend(targets)
        print(f"Loss: {test_metrics['test_loss']:.4f}")
        save_results(fold + 1, test_metrics, config_test,config_test_paths,preds, targets, probs, imgs)


In [5]:
if __name__ == "__main__":
    # Load configs and run main function

    config_test = ConfigTest()
    config_test_paths = ConfigTestPaths()
    config_train = load_config(config_test_paths)

    main(config_train,config_test,config_test_paths)

Test 592
UTAE(
  (in_conv): ConvBlock(
    (conv): ConvLayer(
      (conv): Sequential(
        (0): Conv2d(5, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
        (1): GroupNorm(8, 32, eps=1e-05, affine=True)
        (2): ReLU()
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
        (4): GroupNorm(8, 32, eps=1e-05, affine=True)
        (5): ReLU()
      )
    )
  )
  (down_blocks): ModuleList(
    (0): DownConvBlock(
      (down): ConvLayer(
        (conv): Sequential(
          (0): Conv2d(32, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), padding_mode=reflect)
          (1): GroupNorm(8, 32, eps=1e-05, affine=True)
          (2): ReLU()
        )
      )
      (conv1): ConvLayer(
        (conv): Sequential(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
          (1): GroupNorm(8, 32, eps=1e-05, affine=True)
          (2): ReLU()
     